# EcoHome Energy Advisor - Database Setup

The database stores energy usage (consumption, device types, costs) and solar generation (production, weather).

## Learning Objectives
- Create SQLite database with proper schema
- Populate database with sample data
- Query data for analysis


## 1. Import Required Libraries


In [1]:
import os, sys
from pathlib import Path

ROOT = Path("/Users/sandipdey2/Downloads/udacityprojects/langchain/langgraphenergy/ecohome_solution")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("cwd:", Path.cwd())
print("models exists:", (ROOT / "models" / "energy.py").exists())

cwd: /Users/sandipdey2/Downloads/udacityprojects/langchain/langgraphenergy/ecohome_solution
models exists: True


In [2]:
from datetime import datetime, timedelta
import random
from models.energy import DatabaseManager


## 2. Initialize Database Manager


In [3]:
db_manager = DatabaseManager("data/energy_data.db")


## 3. Create Database Tables


In [4]:
db_manager.create_tables()


Database tables created at /tmp/energy_data.db


## 4. Generate Sample Energy Usage Data


In [5]:
# Generate 30 days of EV / HVAC / appliance usage.
# Bulk-inserted so the notebook stays fast on constrained filesystems.
device_types = {
    'EV': {'base_kwh': 10, 'variation': 5, 'peak_hours': [18, 19, 20, 21]},
    'HVAC': {'base_kwh': 2, 'variation': 1, 'peak_hours': [12, 13, 14, 15, 16, 17]},
    'appliance': {'base_kwh': 1.5, 'variation': 0.5, 'peak_hours': [19, 20, 21, 22]},
}
device_names = {
    'EV': 'Tesla Model 3',
    'HVAC': 'Main AC Unit',
    'appliance': 'Dishwasher',
}

start_date = datetime.now() - timedelta(days=30)
rows = []
rng = random.Random(42)
for day in range(30):
    current_date = start_date + timedelta(days=day)
    for hour in range(24):
        timestamp = current_date.replace(hour=hour, minute=0, second=0, microsecond=0)
        for device_type, config in device_types.items():
            variation = rng.uniform(-config['variation'], config['variation'])
            peak_multiplier = 1.5 if hour in config['peak_hours'] else 0.8
            consumption = max(0, (config['base_kwh'] + variation) * peak_multiplier)
            price_per_kwh = 0.15 if hour in config['peak_hours'] else 0.10
            name = device_names[device_type]
            if device_type == 'appliance':
                name = rng.choice(['Dishwasher', 'Washing Machine', 'Dryer'])
            rows.append({
                'timestamp': timestamp,
                'consumption_kwh': consumption,
                'device_type': device_type,
                'device_name': name,
                'cost_usd': consumption * price_per_kwh,
            })

db_manager.add_usage_records(rows)
print(f"Created {len(rows)} energy usage records")



Created 2160 energy usage records


## 5. Generate Sample Solar Generation Data


In [6]:
weather_conditions = {
    'sunny': {'multiplier': 1.0, 'probability': 0.4},
    'partly_cloudy': {'multiplier': 0.6, 'probability': 0.3},
    'cloudy': {'multiplier': 0.3, 'probability': 0.2},
    'rainy': {'multiplier': 0.1, 'probability': 0.1},
}
start_date = datetime.now() - timedelta(days=30)
rows = []
rng = random.Random(7)
for day in range(31):
    current_date = start_date + timedelta(days=day)
    weather_choice = rng.choices(
        list(weather_conditions),
        weights=[weather_conditions[w]['probability'] for w in weather_conditions],
    )[0]
    weather_multiplier = weather_conditions[weather_choice]['multiplier']
    for hour in range(24):
        timestamp = current_date.replace(hour=hour, minute=0, second=0, microsecond=0)
        if 6 <= hour <= 18:
            hour_factor = 1 - abs(hour - 12) / 6
            generation = max(0, 5.0 * hour_factor * weather_multiplier * rng.uniform(0.8, 1.2))
            base_temp = 20 + rng.uniform(-5, 5)
            irradiance = 800 * hour_factor * weather_multiplier if generation > 0 else 0
            rows.append({
                'timestamp': timestamp,
                'generation_kwh': generation,
                'weather_condition': weather_choice,
                'temperature_c': base_temp,
                'solar_irradiance': irradiance,
            })
db_manager.add_generation_records(rows)
print(f"Created {len(rows)} solar generation records")




Created 403 solar generation records


## 6. Query and Analyze Data


In [7]:
recent_usage = db_manager.get_recent_usage(24)
recent_generation = db_manager.get_recent_generation(24)
print("=== Energy Usage Analysis ===")
print(f"Total records in last 24 hours: {len(recent_usage)}")
device_consumption = {}
for record in recent_usage:
    device = record.device_type or 'unknown'
    device_consumption.setdefault(device, {'kwh': 0, 'cost': 0, 'records': 0})
    device_consumption[device]['kwh'] += record.consumption_kwh
    device_consumption[device]['cost'] += record.cost_usd or 0
    device_consumption[device]['records'] += 1
print("\nConsumption by device type:")
for device, data in device_consumption.items():
    print(f"  {device}: {data['kwh']:.2f} kWh, ${data['cost']:.2f}, {data['records']} records")
print(f"\n=== Solar Generation Analysis ===")
print(f"Total generation records in last 24 hours: {len(recent_generation)}")
print(f"Total generation: {sum(r.generation_kwh for r in recent_generation):.2f} kWh")



=== Energy Usage Analysis ===
Total records in last 24 hours: 54

Consumption by device type:
  EV: 243.07 kWh, $30.76, 18 records
  HVAC: 28.31 kWh, $2.83, 18 records
  appliance: 32.18 kWh, $4.58, 18 records

=== Solar Generation Analysis ===
Total generation records in last 24 hours: 39
Total generation: 90.84 kWh


## 7. Test Database Tools


In [8]:
from datetime import datetime, timedelta
from tools import query_energy_usage, query_solar_generation, get_recent_energy_summary

end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
print("=== Testing Database Tools ===")
print(f"Querying data from {start_date} to {end_date}")

usage_data = query_energy_usage.invoke({"start_date": start_date, "end_date": end_date})
print(f"\nEnergy Usage Query Results:")
print(f"  Total records: {usage_data['total_records']}")
print(f"  Total consumption: {usage_data['total_consumption_kwh']} kWh")
print(f"  Total cost: ${usage_data['total_cost_usd']}")

generation_data = query_solar_generation.invoke({"start_date": start_date, "end_date": end_date})
print(f"\nSolar Generation Query Results:")
print(f"  Total records: {generation_data['total_records']}")
print(f"  Total generation: {generation_data['total_generation_kwh']} kWh")
print(f"  Average daily: {generation_data['average_daily_generation']} kWh")

summary = get_recent_energy_summary.invoke({"hours": 24})
print(f"\nRecent Energy Summary:")
print(f"  Usage: {summary['usage']['total_consumption_kwh']} kWh, ${summary['usage']['total_cost_usd']}")
print(f"  Generation: {summary['generation']['total_generation_kwh']} kWh")
print(f"  Weather: {summary['generation']['average_weather']}")



=== Testing Database Tools ===
Querying data from 2026-08-25 to 2026-09-01

Energy Usage Query Results:
  Total records: 3024
  Total consumption: 12867.53 kWh
  Total cost: $1474.56

Solar Generation Query Results:
  Total records: 494
  Total generation: 757.53 kWh
  Average daily: 94.69 kWh

Recent Energy Summary:
  Usage: 303.56 kWh, $38.17
  Generation: 90.84 kWh
  Weather: sunny
